# faradaymr en Colab (GPU)

Antes de correr: **Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU** (T4 alcanza).

Este notebook:
1. Instala `cupy` para la GPU que Colab haya asignado (se detecta la versión de CUDA automáticamente).
2. Sube/descomprime el proyecto (`faraday.zip`).
3. Corre `run.ejecutar_corrida(use_gpu=True)` -el `sky_map` vectorizado (ver `faradaymr/los_raytrace.py`) hace todo el trabajo pesado en lotes de `PIXEL_CHUNK_SIZE` píxeles por llamada a `map_coordinates`, en vez de un píxel Python a la vez, que es lo que de verdad aprovecha la GPU.
4. Muestra el mapa de cielo ovalado (Mollweide) resultante.

In [ ]:
# --- 1. cupy para la GPU de Colab ---
# Colab trae CUDA preinstalado; el paquete cupy correcto depende de esa
# versión (12.x en las imágenes de Colab recientes). `nvidia-smi` confirma
# que efectivamente hay GPU asignada antes de gastar tiempo instalando.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q cupy-cuda12x

In [ ]:
# --- 2. Subir faraday.zip (el proyecto completo) y descomprimirlo ---
from google.colab import files
subido = files.upload()  # seleccionar faraday.zip en el diálogo

import zipfile, os
nombre_zip = next(iter(subido))
with zipfile.ZipFile(nombre_zip) as z:
    z.extractall('/content/faradaymr_proyecto')

%cd /content/faradaymr_proyecto
!pip install -q -r requirements.txt

In [ ]:
# --- 3. Confirmar que cupy ve la GPU antes de correr nada largo ---
import cupy as cp
print('cupy ve', cp.cuda.runtime.getDeviceCount(), 'GPU(s):', cp.cuda.Device(0).compute_capability)

In [ ]:
# --- 4. Correr el pipeline completo en GPU ---
# `use_gpu=True` (no None) para que falle explícito si por algún motivo
# cupy no quedó disponible, en vez de caer en silencio a CPU -ver
# `faradaymr.backend.get_backend`.
import run
rm_map, i_map, q_map, u_map = run.ejecutar_corrida(use_gpu=True)

In [ ]:
# --- 5. Mostrar el mapa ovalado (Mollweide) generado ---
from IPython.display import Image, display
display(Image('results/foreground_galactico/figura_1_mapa_de_cielo.png'))
display(Image('results/foreground_galactico/figura_2_perfil_latitud.png'))

## Notas de rendimiento / paralelismo

- `config_fisica.PIXEL_CHUNK_SIZE` controla cuántos píxeles (l, b) se resuelven en un solo lote vectorizado dentro de `faradaymr/los_raytrace.py::sky_map`. Subir este número (p. ej. a 16384 o más en una GPU con más memoria, tipo A100) reduce el número de lanzamientos de kernel y acelera la corrida; bajarlo si aparece un error de memoria (`OutOfMemoryError` de cupy).
- Para correr un mapa de mayor resolución (más `N_L`/`N_B` en `config_fisica.py`, el análogo de un `NSIDE` más fino en HEALPix), solo hace falta subir esos números -el `sky_map` vectorizado escala en tiempo con el número de píxeles, no con un bucle de Python por píxel.
- El log de la corrida (`results/logs/`) imprime el tiempo real de `sky_map` (sincronizado con `cupy.cuda.Stream.null.synchronize()`, no solo el tiempo de encolar los kernels) y los dos chequeos de calibración (RM y DM hacia el polo galáctico, ver `faradaymr/calibration.py`).